In [1]:
from datasets import load_dataset

# Load the BASEPROD training split as a stream so the notebook does not download it all at once.
ds = load_dataset("hassanjbara/BASEPROD", split="train", streaming=True)
print(ds)


c:\Users\Ipsita\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IterableDataset({
    features: ['color', 'depth', 'depth_16bit', 'thermal', 'thermal_rgb'],
    num_shards: 106
})


In [2]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm  # plain tqdm, not tqdm.notebook -- avoids the IProgress/ipywidgets requirement


# --- Preprocessing ---

def preprocess_color(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize an RGB color image (in its native mode) and scale to [0, 1]."""
    resized = image.convert("RGB").resize(size, Image.BILINEAR)
    return np.array(resized).astype(np.float32) / 255.0


def preprocess_depth(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize a depth map (in its native mode) and normalize to [0, 1].

    Resizing happens on the original PIL image BEFORE casting to float --
    Pillow cannot resize a 3-channel float32 array directly. If the depth
    image has multiple channels (e.g. depth encoded/visualized across RGB),
    it's collapsed to a single channel by averaging.
    """
    resized = image.resize(size, Image.NEAREST)
    arr = np.array(resized).astype(np.float32)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    depth_range = arr.max() - arr.min()
    if depth_range == 0:
        return np.zeros_like(arr)
    return (arr - arr.min()) / depth_range


def preprocess_thermal(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize a thermal image (in its native mode) and scale to [0, 1].

    Same reasoning as preprocess_depth: resize first, cast to float after.
    """
    resized = image.resize(size, Image.BILINEAR)
    arr = np.array(resized).astype(np.float32)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    therm_range = arr.max() - arr.min()
    if therm_range == 0:
        return np.zeros_like(arr)
    return (arr - arr.min()) / therm_range


# --- Feature extraction ---

def extract_color_features(color_arr: np.ndarray) -> dict:
    """Basic color statistics: per-channel mean/std plus a coarse histogram."""
    features = {}
    for i, channel in enumerate(["r", "g", "b"]):
        features[f"color_{channel}_mean"] = float(color_arr[..., i].mean())
        features[f"color_{channel}_std"] = float(color_arr[..., i].std())
    gray = color_arr.mean(axis=-1)
    hist, _ = np.histogram(gray, bins=8, range=(0.0, 1.0))
    for i, count in enumerate(hist):
        features[f"color_hist_bin_{i}"] = int(count)
    return features


def extract_depth_features(depth_arr: np.ndarray) -> dict:
    """Depth statistics: central tendency, spread, and coverage of valid pixels."""
    valid = depth_arr[depth_arr > 0]
    return {
        "depth_mean": float(valid.mean()) if valid.size else 0.0,
        "depth_std": float(valid.std()) if valid.size else 0.0,
        "depth_valid_ratio": float(valid.size / depth_arr.size),
    }


def extract_thermal_features(thermal_arr: np.ndarray) -> dict:
    """Thermal statistics: overall intensity spread as a proxy for hotspot presence."""
    return {
        "thermal_mean": float(thermal_arr.mean()),
        "thermal_std": float(thermal_arr.std()),
        "thermal_max": float(thermal_arr.max()),
    }


def extract_features(sample: dict) -> dict:
    """Run the full preprocessing + feature extraction pipeline on one sample."""
    features = {}
    if sample.get("color") is not None:
        features.update(extract_color_features(preprocess_color(sample["color"])))
    if sample.get("depth") is not None:
        features.update(extract_depth_features(preprocess_depth(sample["depth"])))
    if sample.get("thermal") is not None:
        features.update(extract_thermal_features(preprocess_thermal(sample["thermal"])))
    return features


# --- Run the pipeline ---

N_SAMPLES = 50  # adjust as needed; the dataset is streamed, so this pulls N_SAMPLES on demand

rows = [extract_features(sample) for sample in tqdm(ds.take(N_SAMPLES), total=N_SAMPLES)]
features_df = pd.DataFrame(rows)
print(f"Extracted features for {len(features_df)} samples")
features_df.head()

  0%|          | 0/50 [00:00<?, ?it/s]'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 1s [Retry 1/5].
100%|██████████| 50/50 [04:01<00:00,  7.80it/s]  'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 1s [Retry 1/5].
100%|██████████| 50/50 [07:05<00:00,  8.50s/it]

Extracted features for 50 samples


,color_r_mean,color_r_std,color_g_mean,color_g_std,color_b_mean,color_b_std,color_hist_bin_0,color_hist_bin_1,color_hist_bin_2,color_hist_bin_3,color_hist_bin_4,color_hist_bin_5,color_hist_bin_6,color_hist_bin_7,depth_mean,depth_std,depth_valid_ratio,thermal_mean,thermal_std,thermal_max
0,0.435919,0.118804,0.436712,0.113637,0.446321,0.109040,146,2934,10867,20497,13598,2119,14,1,0.317739,0.235373,0.868064,0.563776,0.147615,1.0
1,0.451372,0.137061,0.446029,0.139844,0.446984,0.137102,6,2507,15114,14470,11880,5867,328,4,0.318372,0.234357,0.890984,0.404910,0.148695,1.0
2,0.427973,0.150134,0.430662,0.141728,0.438292,0.122754,0,2302,19631,11265,11235,5690,53,0,0.406622,0.232719,0.928392,0.465977,0.230668,1.0
3,0.439243,0.153483,0.453461,0.173291,0.461765,0.173105,46,4168,13530,15345,11235,3053,1234,1565,0.319862,0.231387,0.815011,0.680052,0.155578,1.0
4,0.440911,0.143721,0.441115,0.153758,0.443207,0.147932,3,3395,14857,15533,11633,3523,106,1126,0.319611,0.235106,0.851782,0.653582,0.130200,1.0
